<a href="https://colab.research.google.com/github/Deba088/DeepSurfaceClassifier/blob/implement-train-generator/notebook/train_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install logger

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Preparing metadata (setup.py) ... done
  Created wheel for logger: filename=logger-1.4-py3-none-any.whl size=1757 sha256=453b04ab434616e735fa4e5f09c8e26920f52c5f52538021167972eb49f2c249
  Stored in directory: /root/.cache/pip/wheels/fb/19/7b/09fc73f7503166eaf7f31b4aa0095b7f78af2ec0898e1f8312
Successfully built logger


In [19]:
import logging
import os
import random

import cv2
import numpy as np
import tensorflow as tf
from logger import logger
from tensorflow import keras
from tensorflow.keras.preprocessing.image import img_to_array, load_img

In [17]:
__all__ = ["train_generator", "test_generator", "train_data_length", "test_data_length"]

tf.get_logger().setLevel(logging.ERROR)

In [ ]:
# Mount drive
from google.colab import drive
drive.mount("/content/drive")

# Set train data directory
DATASET_DIR = "/content/drive/MyDrive/Google Colab Datasets/rscd_dataset/train"
logger.info(f"Train data location: {DATASET_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[24/Jun/2023 15:36:01] INFO - Train data location: /content/drive/MyDrive/Google Colab Datasets/rscd_dataset/train


In [ ]:
# Get list of class names
CLASS_NAMES = sorted(os.listdir(DATASET_DIR))

# Log number of classes and class names
NUM_CLASSES = len(CLASS_NAMES)
logger.info(f"Number of classes: {NUM_CLASSES}")
logger.info(f"Class names: {CLASS_NAMES}")

FileNotFoundError: ignored

In [11]:
# Define constants
IMG_HEIGHT, IMG_WEIGHT = 224, 224
BATCH_SIZE = 64
TEST_SIZE = 0.3

In [12]:
# Store list of images as (class_name, image_name)
IMG_LIST = []

for class_name in CLASS_NAMES:
  img_dir = os.path.join(DATASET_DIR, class_name)
  for img in os.listdir(img_dir):
    IMG_LIST.append((img, class_name))

IMG_LIST = np.array(IMG_LIST)

# Shuffle the list of images
np.random.shuffle(IMG_LIST)

NameError: ignored

In [13]:
# Split into train and test sets
train_data, test_data = [], []
for i in IMG_LIST:
  if random.random() < TEST_SIZE:
    test_data.append(i)
  else:
    train_data.append(i)

train_data = np.array(train_data)
test_data = np.array(test_data)

train_data_length = len(train_data)
test_data_length = len(test_data)

In [14]:
# Preprocess image
def preprocess_image(frame):
  frame = cv2.resize(frame, (IMG_HEIGHT, IMG_WEIGHT))
  frame = img_to_array(frame)
  frame = frame / 255.0
  return frame

In [15]:
# Load data into train and test data in the required format
def data_generator(data, batch_size=BATCH_SIZE):
  num_batches = len(data) // batch_size

  for epoch in range(20):
    np.random.shuffle(data)

    # Loop through the num batches
    for batch_number in range(num_batches):
      # Initialize x_batch and y_batch
      x_batch, y_batch = [], []

      for each_data in data[batch_size * batch_number : batch_size * (batch_number + 1)]:
        file_path = os.path.join(os.path.join(DATASET_DIR, each_data[1]), each_data[0])

        # Read image file
        img = cv2.imread(file_path)

        # Preprocess image
        frame = preprocess_image(img)

        x_batch.append(frame)
        y_batch.append(CLASS_NAMES.index(each_data[1]))

        x_batch = np.array(x_batch)
        y_batch = np.array(y_batch)
        y_batch = tf.constant(y_batch)
        y_batch = tf.one_hot(y_batch, depth=NUM_CLASSES)

        yield x_batch, y_batch

In [16]:
train_generator = data_generator(train_data)
test_generator = data_generator(test_data)